# Can we predict `passed_quiz`?
Protocol: `protocol.yaml` (group holdout on `student_key`, 5-fold GroupKFold CV on training rows, choose by mean CV ROC AUC, score holdout once).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='ff08c7a88fd7e15b17b05a390393550fdf57ea2b6df584f1d0594cc2ec1f00e6'
df=pd.read_csv('data.csv')
hold_idx=np.sort(pd.read_csv('holdout.csv')['row'].values)
assert len(hold_idx)==2465
is_hold=np.zeros(len(df),bool); is_hold[hold_idx]=True
train=df[~is_hold]; hold=df[is_hold]
assert not set(train.student_key)&set(hold.student_key)
print(len(train),len(hold),train.passed_quiz.mean())

## Feature choice (prediction time)
Goal: predict the quiz outcome before the quiz that closes the session.
- `student_key`: excluded (entity key, per protocol).
- `session_minutes`: **excluded**. It is the length of the whole session, which the closing quiz is part of, so it is only final once the quiz is done.
- `hints_used`: kept; hints are opened during the session, before the closing quiz.
- `prior_score` (placement test) and `enrol_year`: kept, known beforehand.
- `school_code`: kept, treated as categorical in one candidate and left out of the others.

This is a judgement call; the data dictionary does not say exactly when `session_minutes` is recorded.

In [ ]:
FEATURES=['enrol_year','prior_score','hints_used','school_code']
NUM=['enrol_year','prior_score','hints_used']
def pre(cat=False):
    t=[('num',StandardScaler(),NUM)]
    if cat: t.append(('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=20),['school_code']))
    return ColumnTransformer(t)
# all candidates named before any scoring
CANDIDATES={
 'baseline_prior':lambda: Pipeline([('p',pre()),('m',DummyClassifier(strategy='prior'))]),
 'logreg':lambda: Pipeline([('p',pre()),('m',LogisticRegression(max_iter=1000))]),
 'logreg_school':lambda: Pipeline([('p',pre(True)),('m',LogisticRegression(max_iter=1000))]),
 'rf':lambda: Pipeline([('p',pre()),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=1))]),
 'hgb':lambda: Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0))]),
}
print(list(CANDIDATES))

In [ ]:
X=train[FEATURES]; y=train.passed_quiz.values; g=train.student_key.values
folds=list(GroupKFold(n_splits=5).split(X,y,g))
cv_scores={}
for name,mk in CANDIDATES.items():
    s=[]
    for tr,va in folds:
        m=mk().fit(X.iloc[tr],y[tr])   # preprocessing fitted inside fold
        s.append(roc_auc_score(y[va],m.predict_proba(X.iloc[va])[:,1]))
    cv_scores[name]=s
cv=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv.sort_values('mean',ascending=False)

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen:',chosen)
final=CANDIDATES[chosen]().fit(X,y)
hp=final.predict_proba(hold[FEATURES])[:,1]
holdout_auc=roc_auc_score(hold.passed_quiz,hp)   # scored once
print('holdout roc_auc',holdout_auc)

In [ ]:
train_pos=np.where(~is_hold)[0]
evidence=dict(
 features=FEATURES,
 candidates=list(CANDIDATES),
 cv_folds=[[[int(train_pos[i]) for i in tr],[int(train_pos[i]) for i in va]] for tr,va in folds],
 cv_scores={k:[float(x) for x in v] for k,v in cv_scores.items()},
 chosen=chosen,
 holdout_rows=[int(i) for i in hold_idx],
 holdout_scores={chosen:float(holdout_auc)},
)
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above: values near 0.5 mean the pre-quiz features carry little predictive signal.